# Customer Segmentation for E-Commerce Churn Prevention

**Author:** Yenlik Gaisina | Data & Analytics Consultant | Cambridge Data Science with ML & AI
**Methods:** K-Means Clustering | PCA | t-SNE | RFM Feature Engineering
**Dataset:** UCI Online Retail (publicly available)
**Objective:** Identify actionable customer segments to enable targeted retention, upsell, and win-back campaigns

---

## Table of Contents

1. [Business Context](#1-business-context)
2. [Data Import & Cleaning](#2-data)
3. [Feature Engineering (RFM+)](#3-features)
4. [Exploratory Data Analysis](#4-eda)
5. [Optimal Cluster Selection](#5-cluster-selection)
6. [K-Means Clustering](#6-kmeans)
7. [Dimensionality Reduction & Visualisation](#7-visualisation)
8. [Cluster Profiling & Business Recommendations](#8-profiling)
9. [Reflection & Extensions](#9-reflection)

## 1. Business Context

### Problem Statement

Understanding customer behaviour is critical for e-commerce retention strategy. Rather than treating all customers identically, segmentation enables marketing teams to allocate budget efficiently -- investing in loyalty programmes for high-value customers, running win-back campaigns for churning ones, and nurturing growth-potential segments with targeted promotions.

> **Business Question:** Which customer segments in a transnational e-commerce business are at risk of churn, and how can marketing efforts be tailored to retain, nurture, and re-engage them?

### Approach

This analysis engineers RFM-style features (Recency, Frequency, Monetary Value) from raw transactional data, then applies K-Means clustering to identify natural customer groupings. The segmentation is validated using the Elbow Method, Silhouette Scores, and hierarchical clustering, then visualised with PCA and t-SNE.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#e6edf3',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#e6edf3',
    'grid.color': '#21262d',
    'grid.alpha': 0.6,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'figure.dpi': 120,
})

SEED = 42
np.random.seed(SEED)

C_BLUE   = '#388bfd'
C_GREEN  = '#238636'
C_RED    = '#f85149'
C_AMBER  = '#e3b341'
C_PURPLE = '#bc8cff'
C_TEAL   = '#3fb950'
C_WHITE  = '#e6edf3'

CLUSTER_COLORS = [C_BLUE, C_GREEN, C_RED, C_AMBER, C_PURPLE]

print('Libraries loaded successfully')

## 2. Data Import & Cleaning

The UCI Online Retail dataset contains transactional records from a UK-based online retailer (Dec 2010 - Dec 2011). Each row represents a product line item within an invoice.

| Property | Detail |
|----------|--------|
| **Source** | [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/352/online+retail) |
| **Records** | ~541,909 transaction line items |
| **Customers** | ~4,300 unique customers (after cleaning) |
| **Coverage** | UK-based retailer with international customers |

In [ ]:
# ── Download UCI Online Retail dataset ──────────────────────────────────
import os, zipfile, urllib.request

DATA_URL = 'https://archive.ics.uci.edu/static/public/352/online+retail.zip'
ZIP_PATH = 'online_retail.zip'
CSV_CACHE = 'online_retail_clean.csv'

if os.path.exists(CSV_CACHE):
    df_raw = pd.read_csv(CSV_CACHE, parse_dates=['InvoiceDate'])
    print(f'Loaded from cache: {CSV_CACHE} ({len(df_raw):,} rows)')
else:
    print('Downloading UCI Online Retail dataset...')
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        xls_name = [f for f in z.namelist() if f.endswith('.xlsx')][0]
        z.extract(xls_name)
    df_raw = pd.read_excel(xls_name, engine='openpyxl')
    os.remove(xls_name)
    os.remove(ZIP_PATH)
    df_raw.to_csv(CSV_CACHE, index=False)
    print(f'Downloaded and cached: {len(df_raw):,} rows')

print(f'\nShape: {df_raw.shape}')
print(f'Date range: {df_raw["InvoiceDate"].min()} to {df_raw["InvoiceDate"].max()}')
print(f'Unique customers: {df_raw["CustomerID"].nunique()}')
df_raw.head()

In [ ]:
# ── Data Cleaning ──────────────────────────────────────────────────────
print('=== Missing Values ===')
print(df_raw.isnull().sum())
print(f'\nRows before cleaning: {len(df_raw):,}')

# Drop rows without CustomerID (cannot segment anonymous transactions)
df = df_raw.dropna(subset=['CustomerID']).copy()
df['CustomerID'] = df['CustomerID'].astype(int)

# Remove cancelled orders (InvoiceNo starting with 'C')
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove zero/negative quantity and price
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Remove duplicates
n_dupes = df.duplicated().sum()
df = df.drop_duplicates()

# Calculate line total
df['LineTotal'] = df['Quantity'] * df['UnitPrice']

print(f'\nRows after cleaning: {len(df):,}')
print(f'Duplicates removed: {n_dupes:,}')
print(f'Unique customers: {df["CustomerID"].nunique():,}')
print(f'Unique invoices: {df["InvoiceNo"].nunique():,}')

## 3. Feature Engineering (RFM+)

Raw transactional data must be aggregated to one row per customer. Five behavioural features are engineered:

| Feature | Description | Business Meaning |
|---------|-------------|-----------------|
| **Recency** | Days since last purchase | Current engagement level |
| **Frequency** | Number of unique orders | Loyalty and repeat behaviour |
| **Monetary (CLV)** | Total revenue generated | Customer lifetime value |
| **Avg Unit Price** | Mean price per item purchased | Spending preference tier |
| **Basket Size** | Mean items per order | Purchase behaviour pattern |

In [ ]:
# ── RFM+ Feature Engineering ───────────────────────────────────────────
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

customers = df.groupby('CustomerID').agg(
    recency=('InvoiceDate', lambda x: (reference_date - x.max()).days),
    frequency=('InvoiceNo', 'nunique'),
    monetary=('LineTotal', 'sum'),
    avg_unit_price=('UnitPrice', 'mean'),
    basket_size=('Quantity', lambda x: x.sum() / df.loc[x.index, 'InvoiceNo'].nunique()),
).reset_index()

# Remove extreme outliers (top 1% in monetary to avoid skew from wholesale buyers)
q99 = customers['monetary'].quantile(0.99)
customers = customers[customers['monetary'] <= q99]

print(f'Customer-level dataset: {len(customers):,} customers')
print(f'\nFeature summary:')
print(customers[['recency', 'frequency', 'monetary', 'avg_unit_price', 'basket_size']].describe().round(2))

## 4. Exploratory Data Analysis

In [ ]:
# ── Feature distributions ──────────────────────────────────────────────
features = ['recency', 'frequency', 'monetary', 'avg_unit_price', 'basket_size']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(features):
    ax = axes[i]
    ax.hist(customers[col], bins=50, color=C_BLUE, alpha=0.8, edgecolor='none')
    ax.axvline(customers[col].median(), color=C_RED, lw=2, ls='--',
               label=f'Median: {customers[col].median():.1f}')
    ax.set_title(col.replace('_', ' ').title())
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].axis('off')
fig.suptitle('Feature Distributions (Customer Level)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ────────────────────────────────────────────────
corr = customers[features].corr()
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, linecolor='#30363d',
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Optimal Cluster Selection

Three complementary methods are used to determine the optimal number of clusters:
1. **Elbow Method** -- diminishing WCSS improvement
2. **Silhouette Score** -- cluster separation quality
3. **Hierarchical Clustering** -- dendrogram visual inspection

In [ ]:
# ── Standardise features ───────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(customers[features])

# ── Elbow Method + Silhouette Scores ──────────────────────────────────
K_range = range(2, 11)
wcss = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=SEED)
    km.fit(X_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Elbow plot
ax = axes[0]
ax.plot(K_range, wcss, 'o-', color=C_BLUE, linewidth=2, markersize=6)
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Within-Cluster Sum of Squares (WCSS)')
ax.set_title('Elbow Method')
ax.grid(True, alpha=0.3)

# Silhouette plot
ax = axes[1]
ax.plot(K_range, sil_scores, 'o-', color=C_GREEN, linewidth=2, markersize=6)
best_k = list(K_range)[np.argmax(sil_scores)]
ax.axvline(best_k, color=C_RED, ls='--', lw=1.5, label=f'Best k={best_k}')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Analysis')
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle('Optimal Cluster Selection', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Silhouette scores by k:')
for k, s in zip(K_range, sil_scores):
    marker = ' <-- best' if k == best_k else ''
    print(f'  k={k}: {s:.4f}{marker}')

In [ ]:
# ── Hierarchical Clustering Dendrogram ─────────────────────────────────
# Use a sample for computational efficiency
sample_idx = np.random.choice(len(X_scaled), size=min(1000, len(X_scaled)), replace=False)
X_sample = X_scaled[sample_idx]

linkage_matrix = linkage(X_sample, method='ward')

fig, ax = plt.subplots(figsize=(13, 5))
dendrogram(linkage_matrix, truncate_mode='lastp', p=30,
           leaf_rotation=90, leaf_font_size=9, ax=ax,
           color_threshold=linkage_matrix[-5, 2])
ax.set_title('Hierarchical Clustering Dendrogram (Ward Linkage, 1000-customer sample)')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Distance')
ax.axhline(y=linkage_matrix[-5, 2], color=C_RED, ls='--', lw=1.5,
           label=f'Cut at k=5')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### Cluster selection observations

The Elbow Method shows diminishing WCSS improvement beyond k=4-5. The Silhouette Score analysis identifies the optimal k. The dendrogram supports a similar number of natural groupings. Based on all three methods, we proceed with the best-performing k.

## 6. K-Means Clustering

In [ ]:
# ── Fit K-Means with optimal k ─────────────────────────────────────────
k_chosen = best_k
km_final = KMeans(n_clusters=k_chosen, init='k-means++', n_init=20, random_state=SEED)
customers['cluster'] = km_final.fit_predict(X_scaled)

print(f'K-Means fitted with k={k_chosen}')
print(f'\nCluster sizes:')
for c in range(k_chosen):
    n = (customers['cluster'] == c).sum()
    print(f'  Cluster {c}: {n:,} customers ({n/len(customers)*100:.1f}%)')

print(f'\nSilhouette score: {silhouette_score(X_scaled, customers["cluster"]):.4f}')

In [ ]:
# ── Cluster profiles: boxplots ─────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(features):
    ax = axes[i]
    data = [customers.loc[customers['cluster'] == c, col].values for c in range(k_chosen)]
    bp = ax.boxplot(data, patch_artist=True, widths=0.6,
                    medianprops=dict(color=C_WHITE, linewidth=2))
    for patch, color in zip(bp['boxes'], CLUSTER_COLORS[:k_chosen]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticklabels([f'C{c}' for c in range(k_chosen)])
    ax.set_title(col.replace('_', ' ').title())
    ax.grid(True, alpha=0.3, axis='y')

axes[-1].axis('off')
fig.suptitle('Feature Distributions by Cluster', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. Dimensionality Reduction & Visualisation

PCA and t-SNE are used to project the 5-dimensional feature space into 2D for visual validation of cluster separation.

In [ ]:
# ── PCA ────────────────────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
for c in range(k_chosen):
    mask = customers['cluster'] == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=CLUSTER_COLORS[c],
               label=f'Cluster {c}', alpha=0.5, s=15, edgecolors='none')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('PCA Projection: Customer Segments')
ax.legend(markerscale=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Explained variance: PC1={pca.explained_variance_ratio_[0]:.3f}, PC2={pca.explained_variance_ratio_[1]:.3f}')
print(f'Total explained: {sum(pca.explained_variance_ratio_):.3f}')

In [ ]:
# ── t-SNE ──────────────────────────────────────────────────────────────
# Use a sample for computational efficiency
tsne_n = min(5000, len(X_scaled))
tsne_idx = np.random.choice(len(X_scaled), size=tsne_n, replace=False)
X_tsne_input = X_scaled[tsne_idx]
tsne_labels = customers['cluster'].values[tsne_idx]

tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
X_tsne = tsne.fit_transform(X_tsne_input)

fig, ax = plt.subplots(figsize=(10, 7))
for c in range(k_chosen):
    mask = tsne_labels == c
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=CLUSTER_COLORS[c],
               label=f'Cluster {c}', alpha=0.6, s=15, edgecolors='none')

ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title(f't-SNE Projection ({tsne_n:,} customers)')
ax.legend(markerscale=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Cluster Profiling & Business Recommendations

Each cluster is profiled by its median feature values to identify actionable segment characteristics.

In [ ]:
# ── Cluster profiles ───────────────────────────────────────────────────
profile = customers.groupby('cluster')[features].median().round(2)
profile['n_customers'] = customers.groupby('cluster').size()
profile['pct'] = (profile['n_customers'] / len(customers) * 100).round(1)

print('CLUSTER PROFILES (median values)')
print('=' * 80)
print(profile.to_string())
print()

# Rank clusters by monetary value
ranked = profile.sort_values('monetary', ascending=False)
print('\nClusters ranked by median CLV (monetary):')
for idx, row in ranked.iterrows():
    print(f'  Cluster {idx}: median CLV = {row["monetary"]:,.0f}, '
          f'recency = {row["recency"]:.0f} days, '
          f'frequency = {row["frequency"]:.0f} orders, '
          f'n = {row["n_customers"]:,.0f} ({row["pct"]}%)')

In [ ]:
# ── Assign business labels based on profile characteristics ────────────
# Sort clusters by monetary value to assign labels
profile_sorted = profile.sort_values('monetary', ascending=False)
cluster_order = profile_sorted.index.tolist()

# Assign descriptive labels based on relative position
label_templates = [
    ('High-Value Loyalists', 'Highest CLV, most engaged. Invest in loyalty programmes and exclusive offers.'),
    ('Growth Potential', 'Moderate spending with room to grow. Target with personalised promotions and cross-sell.'),
    ('Steady Core', 'Reliable mid-tier customers. Maintain with consistent service and seasonal campaigns.'),
    ('Occasional Buyers', 'Low frequency but active. Encourage repeat purchases with targeted incentives.'),
    ('At-Risk / Lapsed', 'Highest recency, lowest engagement. Priority for win-back campaigns.'),
]

# Use as many labels as there are clusters
n_labels = min(k_chosen, len(label_templates))
cluster_labels = {}
for i, c in enumerate(cluster_order[:n_labels]):
    cluster_labels[c] = label_templates[i]

print('SEGMENT LABELS & RECOMMENDATIONS')
print('=' * 60)
for c in range(k_chosen):
    if c in cluster_labels:
        label, action = cluster_labels[c]
        n = (customers['cluster'] == c).sum()
        pct = n / len(customers) * 100
        print(f'\nCluster {c}: {label}')
        print(f'  Customers: {n:,} ({pct:.1f}%)')
        print(f'  Action: {action}')
        med = profile.loc[c]
        print(f'  Profile: recency={med["recency"]:.0f}d, frequency={med["frequency"]:.0f}, CLV={med["monetary"]:,.0f}')

In [ ]:
# ── Executive Dashboard ────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Panel 1: Segment size pie
ax1 = fig.add_subplot(gs[0, 0])
sizes = [len(customers[customers['cluster'] == c]) for c in range(k_chosen)]
labels_pie = [f'C{c}' for c in range(k_chosen)]
wedges, texts, autotexts = ax1.pie(
    sizes, labels=labels_pie, colors=CLUSTER_COLORS[:k_chosen],
    autopct='%1.0f%%', startangle=90,
    wedgeprops={'edgecolor': '#0d1117', 'linewidth': 2},
    textprops={'color': C_WHITE, 'fontsize': 9},
)
for at in autotexts:
    at.set_color('#0d1117')
    at.set_fontweight('bold')
ax1.set_title('Segment Distribution')

# Panel 2: Recency vs Frequency scatter
ax2 = fig.add_subplot(gs[0, 1])
for c in range(k_chosen):
    mask = customers['cluster'] == c
    ax2.scatter(customers.loc[mask, 'recency'], customers.loc[mask, 'frequency'],
                c=CLUSTER_COLORS[c], alpha=0.4, s=10, label=f'C{c}')
ax2.set_xlabel('Recency (days)')
ax2.set_ylabel('Frequency (orders)')
ax2.set_title('Recency vs Frequency')
ax2.legend(fontsize=7, markerscale=3)
ax2.grid(True, alpha=0.3)

# Panel 3: CLV by segment
ax3 = fig.add_subplot(gs[0, 2])
medians_clv = [customers.loc[customers['cluster'] == c, 'monetary'].median() for c in range(k_chosen)]
bars = ax3.bar([f'C{c}' for c in range(k_chosen)], medians_clv,
               color=CLUSTER_COLORS[:k_chosen], alpha=0.85, width=0.5)
for bar, val in zip(bars, medians_clv):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(medians_clv)*0.02,
             f'{val:,.0f}', ha='center', fontsize=9, fontweight='bold', color=C_WHITE)
ax3.set_ylabel('Median CLV (GBP)')
ax3.set_title('Customer Lifetime Value by Segment')
ax3.grid(True, alpha=0.3, axis='y')

# Panel 4: PCA with labels
ax4 = fig.add_subplot(gs[1, 0:2])
for c in range(k_chosen):
    mask = customers['cluster'] == c
    label = cluster_labels.get(c, ('Unknown', ''))[0]
    ax4.scatter(X_pca[mask, 0], X_pca[mask, 1], c=CLUSTER_COLORS[c],
                label=f'C{c}: {label}', alpha=0.4, s=8, edgecolors='none')
ax4.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax4.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax4.set_title('PCA Projection with Segment Labels')
ax4.legend(fontsize=7, markerscale=3, loc='upper right')
ax4.grid(True, alpha=0.3)

# Panel 5: Feature heatmap by cluster
ax5 = fig.add_subplot(gs[1, 2])
# Normalise medians for heatmap
profile_norm = profile[features].copy()
for col in features:
    col_min, col_max = profile_norm[col].min(), profile_norm[col].max()
    if col_max > col_min:
        profile_norm[col] = (profile_norm[col] - col_min) / (col_max - col_min)
    # Invert recency so that low recency = high score
    if col == 'recency':
        profile_norm[col] = 1 - profile_norm[col]

sns.heatmap(profile_norm.T, annot=profile[features].T.round(1), fmt='',
            cmap='YlGnBu', ax=ax5, linewidths=0.5, linecolor='#30363d',
            xticklabels=[f'C{c}' for c in range(k_chosen)],
            yticklabels=[f.replace('_', ' ').title() for f in features],
            cbar=False)
ax5.set_title('Cluster Feature Heatmap')

fig.suptitle('Customer Segmentation Dashboard', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Model Summary ──────────────────────────────────────────────────────
print('MODEL SUMMARY')
print('=' * 40)
print(f'Customers segmented  : {len(customers):,}')
print(f'Features used        : {len(features)}')
print(f'Optimal k            : {k_chosen}')
print(f'Silhouette score     : {silhouette_score(X_scaled, customers["cluster"]):.4f}')
print(f'PCA variance (2 comp): {sum(pca.explained_variance_ratio_)*100:.1f}%')
print(f'\nCluster assignments:')
for c in range(k_chosen):
    n = (customers['cluster'] == c).sum()
    label = cluster_labels.get(c, ('Unassigned', ''))[0]
    print(f'  C{c} ({label}): {n:,} ({n/len(customers)*100:.1f}%)')

## 9. Reflection & Extensions

### What Worked Well

- **RFM+ feature engineering** produced interpretable, business-relevant features from raw transactional data. The addition of average unit price and basket size beyond standard RFM gave richer segmentation.
- **Three-method cluster validation** (Elbow, Silhouette, Dendrogram) builds confidence in the chosen k without relying on a single heuristic.
- **t-SNE** provided clearer visual separation than PCA, confirming that the clusters identified by K-Means represent genuine structure in the data rather than artefacts.

### Limitations

- **K-Means assumes spherical clusters** -- customer segments with irregular shapes may be better captured by DBSCAN or Gaussian Mixture Models.
- **Static segmentation** -- customers move between segments over time. A production system would re-segment monthly or quarterly.
- **Single dataset** -- the UCI Online Retail dataset covers one retailer over one year. Cross-retailer validation would strengthen generalisability.

### Potential Extensions

1. **RFM scoring system** -- assign 1-5 scores per RFM dimension for a simpler, more operationally deployable segmentation
2. **DBSCAN or GMM** -- test density-based or probabilistic clustering to capture non-spherical segment shapes
3. **Customer lifetime value prediction** -- build a supervised model to predict future CLV using the cluster label as a feature
4. **Churn prediction** -- combine segmentation with a time-based churn classifier to identify at-risk customers before they lapse
5. **Streamlit dashboard** -- deploy as an interactive app where marketing teams can explore segments and download customer lists